# Manufacturing Inventory DSS: Paper Purchase Decision
### Optimization-Based DSS: ML prediction model + newsvendor optimizer

**How to use this notebook:**
1. Run all cells in order (Runtime → Run all, or Shift+Enter through each cell).
2. When prompted, upload `master_cotizaciones.csv` and `master_ordenes_ejecutivo.csv`.
3. By default this runs a **DEMO** using a real held-out historical batch, so you can
   compare the recommendation against what actually happened.
4. To run it for a real upcoming month instead, fill in `CURRENT_MONTH_OPS` in the
   cell near the bottom with that month's real open orders, then re-run from that
   cell onward.

## Step 0 — Setup

In [1]:
!pip install -q scikit-learn scipy pandas numpy

In [2]:
import re
import pandas as pd
import numpy as np
from dataclasses import dataclass
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error, r2_score
from scipy.stats import norm

### Upload your two CSV files

In [3]:
try:
    # Running in Google Colab
    from google.colab import files
    print("Upload master_cotizaciones.csv and master_ordenes_ejecutivo.csv")
    uploaded = files.upload()
    Q_PATH = "master_cotizaciones.csv"
    PO_PATH = "master_ordenes_ejecutivo.csv"
except ImportError:
    # Running locally (Jupyter) -- put the CSVs in the same folder as this notebook
    Q_PATH = "master_cotizaciones.csv"
    PO_PATH = "master_ordenes_ejecutivo.csv"
    print("Not running in Colab -- expecting the CSVs in the same folder as this notebook.")

Not running in Colab -- expecting the CSVs in the same folder as this notebook.


## Step 1 — ML prediction model

Predicts **total paper cost per order** from `business_line`, `job_type`, and `cantidad`.

- `business_line`: EDITORIAL vs PUBLICOMERCIAL (EMPAQUE + CAJAS PASTA DURA folded into
  PUBLICOMERCIAL — those segments were too small on their own: 19 and 2 orders).
- `job_type`: derived from the `trabajo` text in `master_ordenes_ejecutivo.csv` (the
  real production record, not the quotes file — a quote's job type can reflect an
  offer that never became a real order).
- `cantidad`: kept as a plain number.
- Target: `insumos_total` × 70% (paper's estimated share of total material cost).
- `repuestos_valor` is **not used** — confirmed to be machine spare parts, unrelated
  to paper usage.

In [4]:
PAPER_SHARE = 0.70  # paper is ~70% of total material cost (business knowledge)

JOB_TYPE_KEYWORDS = [
    ("BOOK", r"libro"),
    ("CATALOG", r"cat[aá]logo"),
    ("FLYER", r"volante|flyers?"),
    ("MAGAZINE", r"revista"),
    ("BOX", r"\bcaja\b"),
    ("LABEL", r"etiqueta"),
    ("BROCHURE", r"folleto|d[ií]ptico|tr[ií]ptico"),
    ("STICKER", r"sticker|adhesivos?\b"),
    ("CARD", r"tarjeta"),
    ("AGENDA", r"agenda|cuaderno"),
    ("POSTER_DISPLAY", r"afiche|banderola|rompetr[aá]fico|banderines|separador|roll\s?up"),
    ("CALENDAR", r"calendario"),
    ("NEWSPAPER", r"peri[oó]dico"),
    ("FOLDER", r"carpetas?\b"),
]

def classify_job_type(desc: str) -> str:
    """Derive a job_type category from the trabajo (job description) text."""
    if not isinstance(desc, str):
        return "OTHER"
    d = desc.lower()
    for label, pattern in JOB_TYPE_KEYWORDS:
        if re.search(pattern, d):
            return label
    return "OTHER"

In [5]:
def load_and_prepare(cotizaciones_path: str, ordenes_path: str) -> pd.DataFrame:
    """
    Build the clean modeling dataset. master_ordenes_ejecutivo is the base
    table (authoritative -- real, completed production orders). The only
    things pulled in from master_cotizaciones are insumos_total (material
    cost) and total (quoted client value), since neither exists in the
    orders file.
    """
    q = pd.read_csv(cotizaciones_path)
    po = pd.read_csv(ordenes_path)

    po = po.copy()
    po["op_num"] = po["numero"].astype(str)

    q_key = q.dropna(subset=["ORDEN_PRODUCCION::numero"]).copy()
    q_key["op_num"] = q_key["ORDEN_PRODUCCION::numero"].astype(str).str.replace(".0", "", regex=False)

    merged = po.merge(q_key[["op_num", "insumos_total", "total"]], on="op_num", how="left")
    merged = merged.dropna(subset=["insumos_total"])
    merged = merged.drop_duplicates(subset=["op_num"])

    merged["business_line"] = merged["linea_negocio"].apply(
        lambda x: "EDITORIAL" if x == "EDITORIAL" else "PUBLICOMERCIAL"
    )
    merged["job_type"] = merged["trabajo"].apply(classify_job_type)
    merged["paper_cost_total"] = merged["insumos_total"] * PAPER_SHARE

    cols = ["numero", "business_line", "job_type", "cantidad", "paper_cost_total", "total"]
    return merged[cols].rename(columns={"total": "quoted_value"}).dropna()


data = load_and_prepare(Q_PATH, PO_PATH)
print(f"Clean modeling dataset: {len(data)} orders")
data.head()

Clean modeling dataset: 1087 orders


,numero,business_line,job_type,cantidad,paper_cost_total,quoted_value
0,24189,PUBLICOMERCIAL,CATALOG,3000,839.87260,2886.000396
1,24190,PUBLICOMERCIAL,FLYER,126000,721.82915,2267.998536
2,24191,EDITORIAL,BOOK,1500,887.09075,3150.002666
3,24194,EDITORIAL,BOOK,1500,895.08545,3075.004819
4,24195,EDITORIAL,BOOK,1500,891.38665,3075.000952


In [6]:
def build_pipeline() -> Pipeline:
    preprocessor = ColumnTransformer(transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), ["business_line", "job_type"]),
        ("num", "passthrough", ["cantidad"]),
    ])
    return Pipeline(steps=[("preprocess", preprocessor), ("model", LinearRegression())])


def split_75_20_5(df: pd.DataFrame, random_state=42):
    """75% train, 20% test (accuracy check), 5% held out as a simulated
    one-month batch to feed into the optimizer."""
    train_df, temp_df = train_test_split(df, test_size=0.25, random_state=random_state)
    test_df, optimizer_df = train_test_split(temp_df, test_size=0.20, random_state=random_state)
    return train_df, test_df, optimizer_df


train_df, test_df, optimizer_df = split_75_20_5(data)
print(f"Train: {len(train_df)} | Test: {len(test_df)} | Demo batch: {len(optimizer_df)}")

FEATURES = ["business_line", "job_type", "cantidad"]
TARGET = "paper_cost_total"

pipe = build_pipeline()
pipe.fit(train_df[FEATURES], train_df[TARGET])

preds_test = pipe.predict(test_df[FEATURES])
mae = mean_absolute_error(test_df[TARGET], preds_test)
r2 = r2_score(test_df[TARGET], preds_test)
print(f"Model check (20% test set): MAE ${mae:,.2f} | R^2 {r2:.4f}")

Train: 815 | Test: 217 | Demo batch: 55
Model check (20% test set): MAE $479.25 | R^2 0.1101


In [7]:
# Coefficients, in plain dollar terms -- explainable, not a black box
encoder = pipe.named_steps["preprocess"].named_transformers_["cat"]
cat_names = encoder.get_feature_names_out(["business_line", "job_type"])
all_feature_names = list(cat_names) + ["cantidad"]
coefs = pipe.named_steps["model"].coef_
intercept = pipe.named_steps["model"].intercept_

print(f"Baseline (intercept): ${intercept:,.2f}")
for name, coef in sorted(zip(all_feature_names, coefs), key=lambda x: -abs(x[1])):
    print(f"  {name:35s} {coef:+,.4f}")

Baseline (intercept): $695.74
  business_line_EDITORIAL             +174.2901
  business_line_PUBLICOMERCIAL        -174.2901
  job_type_BOOK                       +141.2966
  job_type_FLYER                      -41.8574
  job_type_POSTER_DISPLAY             -29.3987
  job_type_LABEL                      -23.7149
  job_type_BROCHURE                   -22.8033
  job_type_CARD                       -18.5476
  job_type_MAGAZINE                   -15.5938
  job_type_CATALOG                    +14.9158
  job_type_OTHER                      +10.5481
  job_type_STICKER                    -10.5450
  job_type_AGENDA                     +9.6589
  job_type_BOX                        -8.6861
  job_type_FOLDER                     -3.3761
  job_type_NEWSPAPER                  -2.9471
  job_type_CALENDAR                   +1.0506
  cantidad                            +0.0023


## Step 2 — Optimizer (newsvendor-style purchase decision)

Balances two costs:

- **Holding cost**: cost of $1 of excess paper sitting unused
  
- **Stockout cost**: cost of $1 of paper shortfall (production stop risk)

Budget is **not fixed** — it's calculated as 28% (70% paper share × 40% material-related
cost share) of the month's total order value, so it scales with how busy the shop is.

In [16]:
@dataclass
class CostAssumptions:
    holding_cost_rate: float   # cost of $1 of excess paper held for the month
    stockout_cost_rate: float  # cost of $1 of paper shortfall (production stop risk)


@dataclass
class PurchaseConstraints:
    budget: float              # max $ allowed this cycle (calculated, not fixed)
    storage_capacity: float    # max $ of paper stock that can physically be held
    moq: float = 0.0           # supplier minimum order quantity -- unused, set to 0


BUDGET_FACTOR = 0.70 * 0.40  # 28% of the month's total production order value

def monthly_budget(total_op_value_this_month: float) -> float:
    return total_op_value_this_month * BUDGET_FACTOR


def recommend_purchase(mu: float, sigma: float, costs: CostAssumptions,
                        constraints: PurchaseConstraints) -> dict:
    """Recommend a paper purchase amount ($) for this cycle."""
    if sigma <= 0:
        ideal_purchase = mu
        critical_ratio = None
    else:
        critical_ratio = costs.stockout_cost_rate / (costs.holding_cost_rate + costs.stockout_cost_rate)
        ideal_purchase = mu + sigma * norm.ppf(critical_ratio)

    lower_limit = constraints.moq
    upper_limit = min(constraints.budget, constraints.storage_capacity)

    if lower_limit > upper_limit:
        raise ValueError(
            f"Infeasible: minimum order ({lower_limit}) exceeds "
            f"available budget/storage ({upper_limit})"
        )

    final_purchase = max(lower_limit, min(ideal_purchase, upper_limit))

    binding_constraint = None
    if final_purchase == upper_limit and ideal_purchase > upper_limit:
        binding_constraint = "budget" if constraints.budget < constraints.storage_capacity else "storage"
    elif final_purchase == lower_limit and ideal_purchase < lower_limit:
        binding_constraint = "minimum_order_quantity"

    return {
        "mu": mu, "sigma": sigma, "critical_ratio": critical_ratio,
        "ideal_purchase": ideal_purchase, "recommended_purchase": final_purchase,
        "binding_constraint": binding_constraint or "none (interior optimum)",
    }

# Real business assumptions
# Stockout cost rate -- derived from operating assumptions. 
IDLE_COST_PER_HOUR = 500          # real: cost of one idle production hour
MATERIAL_SHARE = 0.425            # midpoint of 40-45% material-related cost share
PAPER_SHARE_OF_MATERIAL = 0.70    # paper's share of total material cost
HOURS_PER_MONTH = 20 * 22          # normal 8hr/day, 22 working days
avg_monthly_paper_cost = 64144.35  # from 2025 historical monthly average

paper_idle_cost_per_hour = IDLE_COST_PER_HOUR * MATERIAL_SHARE * PAPER_SHARE_OF_MATERIAL
hourly_paper_consumption = avg_monthly_paper_cost / HOURS_PER_MONTH  # from historical data
stockout_cost_rate = paper_idle_cost_per_hour / hourly_paper_consumption

costs = CostAssumptions(holding_cost_rate=0.015, stockout_cost_rate=stockout_cost_rate)
STORAGE_CAPACITY = 300_000 #Fix storage capacity

## Step 3 — This month's orders

Leave `CURRENT_MONTH_OPS` empty to run the **DEMO** (a real held-out historical batch,
so the recommendation can be checked against what actually happened).

To run this for a real upcoming month, fill in the list below with that month's real
open orders, then re-run this cell and the ones after it.

In [17]:
# Example (uncomment and fill in with real orders to switch to LIVE mode):
# CURRENT_MONTH_OPS = [
#     {"order_id": 24189, "business_line": "PUBLICOMERCIAL", "job_type": "CATALOG",
#      "cantidad": 3000, "quoted_value": 2886.00},
#     {"order_id": 24191, "business_line": "EDITORIAL", "job_type": "BOOK",
#      "cantidad": 1500, "quoted_value": 3149.99},
# ]
CURRENT_MONTH_OPS = []

In [18]:
if CURRENT_MONTH_OPS:
    # LIVE MODE: predict for orders whose actual cost isn't known yet
    month_df = pd.DataFrame(CURRENT_MONTH_OPS)
    month_df["predicted_paper_cost"] = pipe.predict(month_df[FEATURES])
    mu = month_df["predicted_paper_cost"].sum()
    residual_std = np.std(test_df[TARGET].values - preds_test)
    sigma = float(residual_std * np.sqrt(len(month_df)))
    total_op_value = month_df["quoted_value"].sum()
    print(f"Mode: LIVE ({len(month_df)} orders from CURRENT_MONTH_OPS)")
    print("(actual cost unknown -- sigma estimated from model's test-set error)")
else:
    # DEMO MODE: use the 5% held-out historical batch (real, known outcome)
    month_df = optimizer_df.copy()
    month_df["predicted_paper_cost"] = pipe.predict(month_df[FEATURES])
    mu = month_df["predicted_paper_cost"].sum()
    actual_total = month_df[TARGET].sum()
    residuals = month_df[TARGET] - month_df["predicted_paper_cost"]
    sigma = float(np.sqrt((residuals ** 2).sum()))
    total_op_value = month_df["quoted_value"].sum()
    print(f"Mode: DEMO ({len(month_df)} real historical orders, held out from training)")
    print(f"Actual paper cost (known, for comparison): ${actual_total:,.2f}")

print(f"Predicted paper need (mu): ${mu:,.2f}")
print(f"Uncertainty (sigma): ${sigma:,.2f}")
print(f"Total order value (for budget calc): ${total_op_value:,.2f}")

Mode: DEMO (55 real historical orders, held out from training)
Actual paper cost (known, for comparison): $50,484.10
Predicted paper need (mu): $42,283.42
Uncertainty (sigma): $9,431.19
Total order value (for budget calc): $160,536.44


In [19]:
budget = monthly_budget(total_op_value)
constraints = PurchaseConstraints(budget=budget, storage_capacity=STORAGE_CAPACITY, moq=0)
result = recommend_purchase(mu, sigma, costs, constraints)

print(f"Dynamic budget (28% of order value): ${budget:,.2f}")
print(f"Storage capacity: ${STORAGE_CAPACITY:,.2f}")
print(f"Ideal purchase (unconstrained):       ${result['ideal_purchase']:,.2f}")
print(f"RECOMMENDED PURCHASE THIS CYCLE:      ${result['recommended_purchase']:,.2f}")
print(f"Binding constraint: {result['binding_constraint']}")

Dynamic budget (28% of order value): $44,950.20
Storage capacity: $300,000.00
Ideal purchase (unconstrained):       $62,879.42
RECOMMENDED PURCHASE THIS CYCLE:      $44,950.20
Binding constraint: budget


## Validation — how would this have compared to reality? *(demo mode only)*

In [20]:
if not CURRENT_MONTH_OPS:
    print(f"Recommended purchase: ${result['recommended_purchase']:,.2f}")
    print(f"What was actually needed: ${actual_total:,.2f}")
    gap = result['recommended_purchase'] - actual_total
    print(f"Gap: ${gap:,.2f} ({gap/actual_total*100:+.1f}%) "
          f"{'(would have covered real need)' if gap >= 0 else '(would have fallen short)'}")
else:
    print("Not applicable -- running in LIVE mode, actual cost isn't known yet.")

Recommended purchase: $44,950.20
What was actually needed: $50,484.10
Gap: $-5,533.89 (-11.0%) (would have fallen short)
